In [11]:
# trapeze_pipeline_parity.py
# Python translation with strict parity to your R logic, plus:
# - The 5th output column is the original imported column C.
# - All transformations/filters still use the true original columns (esp. E).
# Output (no header): trapezerides_test.csv
# Also exposes bill_head(n) to preview top rows.

import os
from pathlib import Path
import pandas as pd
import string

DEBUG = False  # set True to print checkpoints

# ---------- helpers ----------
def win_user_path(tmpl: str) -> str:
    return tmpl.replace("=", os.getlogin())

def set_raw_excel_names(df: pd.DataFrame) -> pd.DataFrame:
    # Mimic readxl col_names=FALSE -> ...1, ...2, ...
    df.columns = [f"...{i+1}" for i in range(df.shape[1])]
    return df

def set_letters_first_26(df: pd.DataFrame) -> pd.DataFrame:
    # After R: colnames(te) <- LETTERS (A..Z for first 26)
    cols = df.columns.tolist()
    for i, ch in enumerate(string.ascii_uppercase[: min(26, len(cols))]):
        cols[i] = ch
    df.columns = cols
    return df

def debug(msg, df=None):
    if DEBUG:
        print(msg)
        if df is not None:
            print(f"rows={len(df)}, cols={list(df.columns)}")
            print(df.head(3))
            print("-"*80)

# ---------- 1) read & pre-clean (R order preserved) ----------
def load_trapeze_excel():
    base = win_user_path(
        r"C:\Users\=\Reconnect Community Health Services\Health Services - Documents\Programs\Transportation\Data\Admin_Transportation\TrapezeData\Exports"
    )
    os.chdir(base)

    te = pd.read_excel("trapeze.xlsx", header=None)
    te = set_raw_excel_names(te)       # ...1, ...2, ...
    te = set_letters_first_26(te)      # A..Z (first 26)

    # filter by I == 9, then set I = "CL"
    te = te[te["I"] == 9].copy()
    te["I"] = "CL"

    # Keep original imported C to force into the 5th output col later
    te["__C_IMPORT__"] = te["C"].copy() if "C" in te.columns else pd.NA

    # Stats.Code init and the R-style column reorder c(1:10,26,11:25)
    te["Stats.Code"] = "a"
    desired = list("ABCDEFGHIJ") + ["Z"] + list("KLMNOPQRSTUVWXY")
    desired = [c for c in desired if c in te.columns]
    te = te.loc[:, desired + [c for c in te.columns if c not in desired]]

    # Round K
    if "K" in te.columns:
        te["K"] = pd.to_numeric(te["K"], errors="coerce").round(2)

    # If X == "TAXI- RCH", set G = "Performed"
    if set(["G", "X"]).issubset(te.columns):
        te.loc[te["X"].eq("TAXI- RCH"), "G"] = "Performed"

    # Map X to "RCH 2"
    if "X" in te.columns:
        te["X"] = te["X"].replace({
            "TAXI- RCH": "RCH 2",
            "TAXI- WN":  "RCH 2",
            "TAXI- SF":  "RCH 2",
            "TAXI - SP": "RCH 2",
        })

    debug("[A] after load & initial reorders", te)
    return te

# ---------- 2) Stats.Code rules (same as R) ----------
def apply_stats_code_rules(te: pd.DataFrame) -> pd.DataFrame:
    for c in ["M", "G", "O"]:
        if c in te.columns:
            te[c] = te[c].astype(str)

    # ADP
    mask_adp = te["M"].isin(["RCH ADS", "HU ADS", "WT ADS", "RCH PRG"]) & te["G"].isin(["Performed", "Scheduled", "Arrived"])
    te.loc[mask_adp, "Stats.Code"] = "TRADS!"
    mask_rch_ads_unknown = (te["M"] == "RCH ADS") & te["G"].isin(["Performed", "Unknown"])
    te.loc[mask_rch_ads_unknown, "Stats.Code"] = "TRADS!"

    mask_tradsfw = (te["M"] == "RCH ADS") & te["G"].isin(["Performed", "Scheduled"]) & (pd.to_numeric(te["K"], errors="coerce") == 0)
    te.loc[mask_tradsfw, "Stats.Code"] = "TRADSFW!"

    mask_tradslx_ads = (
        (te["M"] == "RCH ADS")
        & te["G"].isin(["Unknown Cancellation", "Cancel At Door", "Cancelled Due To Destination Location Being Closed"])
        & pd.to_numeric(te["K"], errors="coerce").isin([4, 4.25, 4.35, 6.90])
    )
    te.loc[mask_tradslx_ads, "Stats.Code"] = "TRADSLX!"

    mask_tradslx_prg = (
        (te["M"] == "RCH PRG")
        & te["G"].isin(["Unknown Cancellation", "Cancel At Door", "Cancelled Due To Destination Location Being Closed"])
        & pd.to_numeric(te["K"], errors="coerce").isin([3.75, 3.85, 6.90])
    )
    te.loc[mask_tradslx_prg, "Stats.Code"] = "TRADSLX!"

    mask_tradsnc1 = (te["M"] == "RCH ADS") & (te["O"] == "Canc. N/C") & (pd.to_numeric(te["K"], errors="coerce") == 0)
    te.loc[mask_tradsnc1, "Stats.Code"] = "TRADSNC!"
    mask_tradsnc2 = (te["M"] == "RCH ADS") & te["G"].isin(["Advance Cancellation", "Cancelled Due To Destination Location Being Closed"])
    te.loc[mask_tradsnc2, "Stats.Code"] = "TRADSNC!"
    mask_tradsfw2 = (te["M"] == "RCH ADS") & (te["O"] == "FareWaived") & (pd.to_numeric(te["K"], errors="coerce") == 0)
    te.loc[mask_tradsfw2, "Stats.Code"] = "TRADSFW!"

    # Rest
    te.loc[(te["M"] == "NORMAL") & (te["G"] == "Advance Cancellation"), "Stats.Code"] = "TRDNC!"

    # helpers
    q = pd.to_numeric(te["Q"], errors="coerce")
    k = pd.to_numeric(te["K"], errors="coerce")

    # 1–5
    base_1_5 = (q <= 5)
    for kval in [3.85, 6.6, 4.35, 6.75, 6.90, 11, 13.8]:
        m = (te["M"] == "NORMAL") & te["G"].isin(["Performed", "Scheduled", "Unknown", "Arrived"]) & base_1_5 & (k == kval)
        te.loc[m, "Stats.Code"] = "TRD1!"
    te.loc[(te["M"] == "NORMAL") & te["G"].isin(["Performed", "Scheduled"]) & base_1_5 & (k == 4), "Stats.Code"] = "TRADS!"
    for kval in [6.6, 13.8, 6.75, 6.90, 13.5, 4, 3.6, 3.85]:
        m = (te["M"] == "NORMAL") & te["G"].isin(
            ["Unknown Cancellation", "Cancel At Door", "Cancelled Due To Destination Location Being Closed"]
        ) & base_1_5 & (k == kval)
        te.loc[m, "Stats.Code"] = "TRD1LX!"

    # 5–10
    base_5_10 = (q > 5) & (q <= 10)
    for kval in [10.5, 10.75, 11, 0, 21.5, 22]:
        m = (te["M"] == "NORMAL") & te["G"].isin(["Performed", "Scheduled", "Arrived"]) & base_5_10 & (k == kval)
        te.loc[m, "Stats.Code"] = "TRD2!"
    for kval in [10.5, 10.75, 11, 0]:
        m = (te["M"] == "NORMAL") & te["G"].isin(
            ["Unknown Cancellation", "Cancel At Door", "Unknown", "Cancelled Due To Destination Location Being Closed"]
        ) & base_5_10 & (k == kval)
        te.loc[m, "Stats.Code"] = "TRD2LX!"

    # 10–15
    base_10_15 = (q > 10) & (q <= 15)
    for kval in [13.2, 13.5]:
        m = (te["M"] == "NORMAL") & te["G"].isin(["Performed", "Scheduled"]) & (q <= 5) & (k == kval)
        te.loc[m, "Stats.Code"] = "TRD3!"
    for kval in [15.25, 15.6, 15.95]:
        m = (te["M"] == "NORMAL") & te["G"].isin(["Performed", "Scheduled"]) & base_10_15 & (k == kval)
        te.loc[m, "Stats.Code"] = "TRD3!"
        m2 = (te["M"] == "NORMAL") & te["G"].isin(["Unknown Cancellation", "Cancel At Door", "Unknown"]) & base_10_15 & (k == kval)
        te.loc[m2, "Stats.Code"] = "TRD3LX!"

    # 15–20
    base_15_20 = (q > 15) & (q <= 20)
    for kval in [18.75, 19.65]:
        m = (te["M"] == "NORMAL") & te["G"].isin(["Performed", "Scheduled"]) & base_15_20 & (k == kval)
        te.loc[m, "Stats.Code"] = "TRD4!"
    for kval in [18.75, 19.25, 19.65]:
        m = (te["M"] == "NORMAL") & te["G"].isin(["Unknown Cancellation", "Cancel At Door"]) & base_15_20 & (k == kval)
        te.loc[m, "Stats.Code"] = "TRD4LX!"

    # 20–25
    base_20_25 = (q > 20) & (q <= 25)
    te.loc[(te["M"] == "NORMAL") & te["G"].isin(["Performed", "Scheduled"]) & base_20_25 & (k == 23), "Stats.Code"] = "TRD5!"
    te.loc[(te["M"] == "NORMAL") & te["G"].isin(["Unknown Cancellation", "Cancel At Door"]) & base_20_25 & (k == 23), "Stats.Code"] = "TRD5LX!"

    # >25
    base_gt_25 = (q > 25)
    te.loc[(te["M"] == "NORMAL") & te["G"].isin(["Performed", "Scheduled"]) & base_gt_25, "Stats.Code"] = "TRD6!"
    te.loc[(te["M"] == "NORMAL") & te["G"].isin(["Unknown Cancellation", "Cancel At Door"]) & base_20_25, "Stats.Code"] = "TRD6LX!"
    te.loc[(te["M"] == "NORMAL") & te["G"].isin(["Unknown Cancellation", "Cancel At Door"]) & base_gt_25, "Stats.Code"] = "TRD6LX!"

    # HAL
    te.loc[pd.to_numeric(te["K"], errors="coerce").eq(26.27), "Stats.Code"] = "TRHAL"

    # EHC/Shopping
    for kval in [3.6, 3.7, 3.85]:
        te.loc[(k == kval) & te["G"].isin(["Performed", "Scheduled"]), "Stats.Code"] = "TRPRG!"
    te.loc[(k == 0.0) & te["G"].isin(["Unknown Cancellation", "Cancel At Door", "Performed"]), "Stats.Code"] = "TRPRGLX!"
    for kval in [3.6, 3.7]:
        te.loc[(k == kval) & te["G"].isin(["Unknown Cancellation", "Cancel At Door"]), "Stats.Code"] = "TRPRGLX!"
    te.loc[(te["M"] == "NORMAL") & (te["O"] == "FareWaived"), "Stats.Code"] = "TRPRGFW!"
    te.loc[(te["M"] == "NORMAL") & te["O"].isin(["Canc. N/C", "No Charge"]), "Stats.Code"] = "TRPRGNC!"

    debug("[B] after Stats.Code rules", te)
    return te

# ---------- 3) column fixes + ESCORT handling (R order!) ----------
def finalize_columns_pre_join(te: pd.DataFrame) -> pd.DataFrame:
    # D padded to 7 chars
    if "D" in te.columns:
        te["D"] = te["D"].astype(str)
        te["D"] = ("000000" + te["D"]).str[-7:]

    # ensure positions 3->"C", 4->"D" labels
    if te.shape[1] >= 4:
        te.columns = [*(te.columns[:2]), "C", "D", *te.columns[4:]]

    # ESCORT adjustments then drop N
    if "N" in te.columns and "K" in te.columns:
        te.loc[te["N"] == "ESCORT", "K"] = 0
        te = te[te["N"] != "ESCORT"].copy()
        te = te.drop(columns=["N"])

    # drop codes
    drop_codes = {"TRADSFW!", "TRADSNC!", "TRDFW!", "TRDNC!", "TRPRGFW!", "TRPRGNC!", "TRSHPFW!", "TRSHPNC!", "TRADSFW!"}
    if "Stats.Code" in te.columns:
        te = te[~te["Stats.Code"].isin(drop_codes)].copy()

    # E replacements
    if "E" in te.columns:
        te["E"] = te["E"].astype(str).replace({
            "000669627": "AC000669627",
            "000005592": "AC000005592",
            "000007326": "AC000007326",
            "000005078": "AC000005078",
            "000007449": "AC000007449",
            "000006174": "AC000006174",
            "000007538": "AC000007538",
            "000006557": "AC000006557",  # will be replaced below to 1017294
            "001096265": "AC001096265",
            "001345":    "0013445",
            "AC000006557":"1017294",
        })

    debug("[C] after pre-join column fixes", te)
    return te

# ---------- 4) join with AlayaCare & rebuild like R ----------
def join_with_alayacare_and_build(te: pd.DataFrame) -> pd.DataFrame:
    alaya_dir = win_user_path(
        r"C:\Users\=\Reconnect Community Health Services\Health Services - Documents\Decision Support\Decision Support Data\Reports\Data\AlayaCare"
    )
    os.chdir(alaya_dir)
    adt = pd.read_csv("ADT with Zone and Reference.csv", dtype=str)

    # adt$ProcuraID <- ifelse(ClientID == "AC000945993", 1014989, ProcuraID)
    if {"ClientID", "ProcuraID"}.issubset(adt.columns):
        adt.loc[adt["ClientID"] == "AC000945993", "ProcuraID"] = "1014989"

    # filter ServiceCodeDept == "Transportation"
    if "ServiceCodeDept" in adt.columns:
        adt = adt[adt["ServiceCodeDept"] == "Transportation"].copy()

    # select c(3,25,27,26) -> 0-based [2,24,26,25], then rename
    take = [2, 24, 26, 25]
    take = [i for i in take if i < adt.shape[1]]
    adt_small = adt.iloc[:, take].copy()
    new_names = ["AID", "PID", "NewID", "Client"]
    adt_small.columns = new_names[: len(adt_small.columns)]

    # NewPID
    pid = adt_small.get("PID", pd.Series([""] * len(adt_small)))
    new_id = adt_small.get("NewID", pd.Series([""] * len(adt_small)))
    adt_small["NewPID"] = pid.fillna("").where(pid.fillna("") != "", new_id.fillna(""))
    adt_small["NewPID"] = "90000" + adt_small["NewPID"].astype(str)
    adt_small = adt_small.drop(columns=[c for c in ["NewID", "Client"] if c in adt_small.columns], errors="ignore")
    adt_small = adt_small.drop_duplicates()

    # split by E starting with "AC"
    is_ac = te["E"].astype(str).str.startswith("AC")
    atrap = te[is_ac].copy()
    ptrap = te[~is_ac].copy()

    # AC: E -> AID, join, and remember original E as __E_FROM_SPLIT__
    if "E" in atrap.columns:
        atrap = atrap.rename(columns={"E": "AID"})
        atrap["__E_FROM_SPLIT__"] = atrap["AID"]  # original E
    if "AID" in atrap.columns and "AID" in adt_small.columns:
        atrap = atrap.merge(adt_small, on="AID", how="left")

    # non-AC: E -> PID, join, and remember original E as __E_FROM_SPLIT__
    if "E" in ptrap.columns:
        ptrap = ptrap.rename(columns={"E": "PID"})
        ptrap["__E_FROM_SPLIT__"] = ptrap["PID"]  # original E
    join_key = "PID" if "PID" in adt_small.columns else "NewPID"
    ptrap = ptrap.merge(adt_small, left_on="PID", right_on=join_key, how="left")

    # rbind
    te2 = pd.concat([ptrap, atrap], ignore_index=True)

    # R positional reorder: 1:4, 27, 26, 5:25  (1-based)
    cols = te2.columns.tolist()
    def pick_pos(pos_1based):
        return [cols[p-1] for p in pos_1based if 0 < p <= len(cols)]
    reorder = pick_pos(list(range(1, 5)) + [27, 26] + list(range(5, 26)))
    reorder = [c for c in reorder if c in cols] + [c for c in cols if c not in reorder]
    te2 = te2.loc[:, reorder]

    # Rebuild a true E column for downstream filters from __E_FROM_SPLIT__
    if "__E_FROM_SPLIT__" in te2.columns:
        te2["E"] = te2["__E_FROM_SPLIT__"].values

    # Force the 5th column's VALUES to be the original imported C
    # (preserved as __C_IMPORT__ through the split/join)
    if "__C_IMPORT__" in te2.columns and te2.shape[1] >= 5:
        te2.iloc[:, 4] = te2["__C_IMPORT__"].values

    # Drop helpers
    for c in ["__E_FROM_SPLIT__"]:
        if c in te2.columns:
            te2 = te2.drop(columns=[c])

    debug("[D] after join + rebuild E + enforce 5th col from original C", te2)
    return te2

# ---------- 5) pure builder: build billable without writing ----------
def build_billable(te2: pd.DataFrame) -> pd.DataFrame:
    """Pure builder: returns the billable DataFrame (no file writes)."""
    bill = te2.copy()

    # Keep rows where true E is NOT NA (parity with R's is.na(E==" "))
    if "E" in bill.columns:
        bill = bill[~bill["E"].isna()].copy()

    # I <- Y; drop Y/AID/PID; then I <- "9"
    if "Y" in bill.columns:
        bill["I"] = bill["Y"]
    bill = bill.drop(columns=[c for c in ["Y", "AID", "PID"] if c in bill.columns], errors="ignore")
    bill["I"] = "9"

    # Positional reorder: c(1:3,5,4,6:24)
    cols = bill.columns.tolist()
    def take_idx(seq):
        return [cols[i-1] for i in seq if 0 < i <= len(cols)]
    order = take_idx(list(range(1,4)) + [5,4] + list(range(6,25)))
    order = [c for c in order if c in bill.columns] + [c for c in bill.columns if c not in order]
    bill = bill.loc[:, order]

    # Drop where 8th column is in {503, 500, 504, 509}
    if bill.shape[1] >= 8:
        col8 = pd.to_numeric(bill.iloc[:, 7], errors="coerce")
        bill = bill[~col8.isin([503, 500, 504, 509])].copy()

    debug("[E] final billable (pure, no write)", bill)
    return bill

# ---------- 6) writer ----------
def build_billable_and_write(te2: pd.DataFrame):
    bill = build_billable(te2)
    out_dir = win_user_path(
        r"C:\Users\=\OneDrive - Reconnect Community Health Services\Desktop\trapezerides_test"
    )
    os.chdir(out_dir)
    bill.to_csv("trapezerides_test.csv", index=False, header=False)
    return bill

# ---------- convenience: just get the head for preview ----------
def bill_head(n: int = 5) -> pd.DataFrame:
    """Build and return the first n rows of the bill (no file writes)."""
    te  = load_trapeze_excel()
    te  = apply_stats_code_rules(te)
    te  = finalize_columns_pre_join(te)
    te2 = join_with_alayacare_and_build(te)
    bill = build_billable(te2)
    return bill.head(n).copy()

# ---------- main ----------
def run():
    te = load_trapeze_excel()
    te = apply_stats_code_rules(te)
    te = finalize_columns_pre_join(te)         # includes ESCORT handling BEFORE dropping N
    te2 = join_with_alayacare_and_build(te)
    bill = build_billable_and_write(te2)
    return bill

if __name__ == "__main__":
    df_out = run()
    print(f"Done. Rows written: {len(df_out)}.")
    # Preview top 10 rows (no headers) for quick visual check
    try:
        preview = bill_head(10)
        with pd.option_context("display.max_columns", None, "display.width", 200, "display.max_colwidth", 200):
            print("\n[Bill Head Preview]")
            print(preview.to_string(index=False, header=False))
    except Exception as e:
        print(f"[Bill Head Preview] skipped: {e}")


Done. Rows written: 290.

[Bill Head Preview]
20250929 3347578  4450 TRPRGLX! 0001833 1 Unknown Cancellation 505 9 DAY PRG 0.00 24 RCH ADS FareWaived 0.00 4.24   NaN NaN   NaN NaN NaN      NaN     NaN  4450 900000001833 0001833
20250929 3347579  4450 TRPRGLX! 0001833 2 Unknown Cancellation 505 9 DAY PRG 0.00 24 RCH ADS  Canc. N/C 0.00 4.27   NaN NaN   NaN NaN NaN      NaN     NaN  4450 900000001833 0001833
20250929 3347580 17121   TRADS! 0001362 1            Performed  30 9 DAY PRG 4.35 24 RCH ADS     ADS SC 0.00 0.70 266.0  HU 454.0 NaN NaN 565661.0   RCH14 17121 900000001362 0001362
20250929 3347581 17121   TRADS! 0001362 2            Performed  30 9 DAY PRG 4.35 24 RCH ADS     ADS SC 0.00 0.70 160.0  HU  95.0 NaN NaN 565715.0   RCH 3 17121 900000001362 0001362
20250929 3347582 24191   TRADS! 0000970 1            Performed  30 9 DAY PRG 4.35 24 RCH ADS     ADS SC 0.00 2.26 315.0 RCH 557.0 NaN NaN 565702.0   RCH 1 24191 900000000970 0000970
20250929 3347583 24191   TRADS! 0000970 2   

In [8]:
bill

NameError: name 'bill' is not defined